# Dataset exploration dashboard

Interactive view of saved HF/LF reaction–diffusion datasets.

- Select a **parameter case** from the dropdown
- Top row: HF spatial fields at the final snapshot
- Bottom row: HF domain mean vs LF trajectory (full history, marker at final time)

## Load dataset

In [1]:
from pathlib import Path
from sklearn.utils.extmath import randomized_svd

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from models.backends import Plotter
from models.config import case_label
from models.loader import load_dataset

dataset_dir = Path("data/reaction_diffusion/rd_kinetics_sources")
data = load_dataset(dataset_dir)

params = data["params"]
times = data["times"]
nodes = data["nodes"]
hf = data["hf"]
lf = data["lf"]
num_species = hf.shape[1]
n_cases = params.shape[0]
n_times = times.shape[0]

plotter = Plotter(nodes)
hf_mean = hf.mean(axis=-1)

print(f"Loaded {dataset_dir}")
print(f"cases={n_cases}, snapshots={n_times}, nodes={nodes.shape[0]}, species={num_species}")
if "timing_hf" in data:
    print(f"HF timing: {data['timing_hf'].mean():.2f} ± {data['timing_hf'].std():.2f} s")
    print(f"LF timing: {data['timing_lf'].mean():.3f} ± {data['timing_lf'].std():.3f} s")

Loaded data/reaction_diffusion/rd_kinetics_sources
cases=500, snapshots=501, nodes=1089, species=6
HF timing: 10.50 ± 0.08 s
LF timing: 0.005 ± 0.001 s


## Parameter case overview

In [2]:
import pandas as pd

overview = pd.DataFrame(params, columns=["linear_scale", "source_t_peak", "source_pulse_width"])
overview.index.name = "case"
if "timing_hf" in data:
    overview["timing_hf_s"] = data["timing_hf"]
    overview["timing_lf_s"] = data["timing_lf"]
overview

,linear_scale,source_t_peak,source_pulse_width,timing_hf_s,timing_lf_s
case,,,,,
0,0.5,0.5,0.2,10.513093,0.004150
1,0.5,0.5,0.4,10.652134,0.003646
2,0.5,0.5,0.6,10.480062,0.003428
3,0.5,0.5,0.8,10.540704,0.003340
4,0.5,0.5,1.0,10.550849,0.004057
...,...,...,...,...,...
495,2.0,3.0,1.2,10.463202,0.005532
496,2.0,3.0,1.4,10.442502,0.005905
497,2.0,3.0,1.6,10.446836,0.005914


## HF vs LF dashboard

Use the widgets below to explore spatial fields and lumped trajectories.

In [ ]:
final_time_idx = n_times - 1
final_time = times[final_time_idx]

case_options = [(case_label(params[i], i), i) for i in range(n_cases)]

case_dropdown = widgets.Dropdown(
    options=case_options,
    description="Case:",
    layout=widgets.Layout(width="95%"),
)
time_readout = widgets.Label(value=f"Snapshot: t = {final_time:.3f} (final)")
plot_out = widgets.Output()


def render_comparison(case_idx: int):
    ls, tp, w = params[case_idx]
    tt = final_time_idx
    t_sel = final_time

    fig, axs = plt.subplots(2, num_species, figsize=(3.5 * num_species, 6), constrained_layout=True)
    if num_species == 1:
        axs = np.array([[axs[0]], [axs[1]]])

    y_max = 0.0
    for i in range(num_species):
        y_max = max(y_max, hf_mean[case_idx, i].max(), lf[case_idx, :, i].max())
    y_max *= 1.1

    for i in range(num_species):
        snap = hf[case_idx, i, tt, :]
        c = plotter.plot(axs[0, i], snap, levels=40, cmap="jet", streamlines=False)
        fig.colorbar(c, ax=axs[0, i], format="%.2f")
        axs[0, i].set_title(f"HF $c_{i}$ at $t={t_sel:.2f}$")

        axs[1, i].plot(times, hf_mean[case_idx, i], "b-", label="HF mean")
        axs[1, i].plot(times, lf[case_idx, :, i], "r--", label="LF (ODE)")
        axs[1, i].axvline(t_sel, color="k", ls=":", lw=1)
        axs[1, i].scatter([t_sel], [hf_mean[case_idx, i, tt]], color="b", s=25, zorder=3)
        axs[1, i].scatter([t_sel], [lf[case_idx, tt, i]], color="r", s=25, zorder=3)
        axs[1, i].set_xlim(times[0], times[-1])
        axs[1, i].set_ylim(0, y_max)
        axs[1, i].set_xlabel("Time")
        if i == 0:
            axs[1, i].set_ylabel("Concentration")

    handles, labels = axs[1, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(
        f"case {case_idx}: linear_scale={ls:g}, source_t_peak={tp:g}, source_pulse_width={w:g}",
        y=1.05,
    )

    with plot_out:
        plot_out.clear_output(wait=True)
        display(fig)
        plt.close(fig)


def _update(_=None):
    render_comparison(case_dropdown.value)


case_dropdown.observe(_update, names="value")

controls = widgets.VBox(
    [
        case_dropdown,
        time_readout,
        plot_out,
    ]
)
display(controls)
_update()

## Solve time summary (if available)

In [ ]:
if "timing_hf" in data:
    labels = ["HF (PDE)", "LF (ODE)"]
    means = [data["timing_hf"].mean(), data["timing_lf"].mean()]
    stds = [data["timing_hf"].std(), data["timing_lf"].std()]

    fig, ax = plt.subplots(figsize=(5, 3.5))
    bars = ax.bar(labels, means, yerr=stds, capsize=6, color=["#1f77b4", "#ff7f0e"], alpha=0.85)
    ax.set_ylabel("Wall time [s]")
    ax.set_title(f"Solve time over {n_cases} cases")
    ax.bar_label(bars, labels=[f"{m:.3f}s" for m in means])
    plt.tight_layout()
else:
    print("No timing data in params.npz — re-run create_dataset.ipynb to record timings.")